# Load CSV data to bronze layer.


## 1. Import csv config data

In [0]:
# import sys module
import sys
sys.path.append("/Workspace/Users/sophonsaesim@gmail.com/ReadCSVdata_2026/readcsvdata/configs")

In [0]:

import bronze_config as bc  # import bronze configuration
import importlib            # import reload bronze config
import pyspark.sql.functions as F # import pyspart function

# refesh bronze configuration
importlib.reload(bc)

## 2. Bronze layer ingestion program

### Function : Read bronze data

In [0]:
# Function : Read bronze data
def read_bronze(config):
    return (
        spark.read.format(config["format"])
                    .option("header", True)
                    .option("inferSchema", True)
                    .load(f"{config["path_land"]}/{config["file"]}.{config["format"]}")
    )

### Function : Write bronze layer

In [0]:
# function : write bronze data
def write_bronze(bronze_df, config):
    return (
        bronze_df.write.mode("overwrite")
                    .option("mergeSchema", True)
                    .saveAsTable(f"{config["table_path"]}.{config["table"]}") 
                    
    )

### Function : Move completed write file


In [0]:
# Move completed write file from land to processed folder
def move_bronze(config):
    load_res = "OK"
    landing_path = f"{config["path_land"]}/{config["file"]}.{config["format"]}"
    process_path = f"{config["path_processed"]}/{config["file"]}.{config["format"]}"
    # move completed write file from land to processed folder
    dbutils.fs.mv(landing_path, process_path)
    return load_res

### Function : Bronze load data main program

In [0]:
def exec_bronze():
    # read data to bronze
    for config in bc.BRONZE_CONFIG:
        bronze_df = read_bronze(config)

        # write bronze data to 
        bronze_df = bronze_df.withColumn("current_time", 
                                         F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss")
                                         )
        write_bronze(bronze_df, config)

        # check existing table after completed ingestion    
        if spark.table(f"{config["table_path"]}.{config["table"]}").count() != 0:
            # move file from land to processed folder
            move_res = move_bronze(config)
            if move_res == "OK":
                print(f"Full bronze data ingestion completed")
            else:
                print(f"Error : {move_res}")

### Execute bronze data ingestion 

In [0]:
exec_bronze = exec_bronze()
if exec_bronze == "OK":
    print("All bronze layer data ingestion completed")
else:
    print("Error : ", exec_bronze)